# 📽️ PLAYWRIGHT - Model Training

This notebook trains a transformer-based sequence labeling model for beat segmentation.

**Model Architecture:**
- Base: DistilBERT (lightweight, fast inference)
- Task: Token classification (BIO tagging for beat boundaries)
- Additional heads: Mood classification, Camera angle prediction

---
**⚠️ IMPORTANT: Enable GPU**
1. Go to `Runtime` → `Change runtime type`
2. Select `T4 GPU`
3. Click `Save`

**Runtime:** ~1-2 hours with GPU

## Setup

In [1]:
!pip install transformers torch datasets accelerate pandas scikit-learn -q

In [2]:
# Check GPU availability
import torch
import sys

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected! Training will be slow.")
    print("Go to Runtime → Change runtime type → Select T4 GPU")

if not hasattr(sys.modules["__main__"], "__file__"):
  sys.modules["__main__"].__file__ = "notebook.py"

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from typing import Dict, List, Optional
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizer,
    DistilBertModel,
    DistilBertPreTrainedModel,
    DistilBertConfig,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm
import transformers

# Paths
DATA_PATH = "/content/drive/MyDrive/playwright_data"
OUTPUT_PATH = f"{DATA_PATH}/processed"
MODEL_PATH = f"{DATA_PATH}/model"

os.makedirs(MODEL_PATH, exist_ok=True)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 1. Model Architecture

In [5]:
# Label mappings
LABEL_TO_ID = {'O': 0, 'B-BEAT': 1, 'I-BEAT': 2}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}

MOOD_TO_ID = {
    'neutral': 0, 'tense': 1, 'romantic': 2, 'action': 3,
    'mysterious': 4, 'comedic': 5, 'dramatic': 6, 'serene': 7, 'melancholic': 8
}
ID_TO_MOOD = {v: k for k, v in MOOD_TO_ID.items()}

CAMERA_TO_ID = {
    'medium shot': 0, 'close-up': 1, 'wide shot': 2, 'over-the-shoulder': 3,
    'pov shot': 4, 'tracking shot': 5, 'low angle': 6, 'high angle': 7
}
ID_TO_CAMERA = {v: k for k, v in CAMERA_TO_ID.items()}

print(f"Labels: {len(LABEL_TO_ID)}")
print(f"Moods: {len(MOOD_TO_ID)}")
print(f"Cameras: {len(CAMERA_TO_ID)}")

Labels: 3
Moods: 9
Cameras: 8


In [6]:
class ScriptSegmentationModel(DistilBertPreTrainedModel):
    """
    Multi-task model for script segmentation.

    Tasks:
    1. Beat boundary detection (BIO tagging)
    2. Mood classification
    3. Camera angle prediction
    """
    _auto_class = None

    def __init__(self, config, num_labels=3, num_moods=9, num_cameras=8):
        super().__init__(config)

        self.num_labels = num_labels
        self.num_moods = num_moods
        self.num_cameras = num_cameras

        # Base encoder
        self.distilbert = DistilBertModel(config)
        self.dropout = nn.Dropout(config.dropout)

        # Task heads
        self.beat_classifier = nn.Linear(config.hidden_size, num_labels)

        self.mood_classifier = nn.Sequential(
            nn.Linear(config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_moods)
        )

        self.camera_classifier = nn.Sequential(
            nn.Linear(config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_cameras)
        )

        self.post_init()

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        mood_labels: Optional[torch.Tensor] = None,
        camera_labels: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:

        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        sequence_output = self.dropout(outputs.last_hidden_state)

        beat_logits = self.beat_classifier(sequence_output)

        pooled_output = sequence_output[:, 0, :]
        mood_logits = self.mood_classifier(pooled_output)
        camera_logits = self.camera_classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

            beat_loss = loss_fct(
                beat_logits.view(-1, self.num_labels),
                labels.view(-1)
            )

            total_loss = beat_loss

            if mood_labels is not None:
                mood_loss = nn.CrossEntropyLoss()(mood_logits, mood_labels)
                total_loss = total_loss + 0.3 * mood_loss

            if camera_labels is not None:
                camera_loss = nn.CrossEntropyLoss()(camera_logits, camera_labels)
                total_loss = total_loss + 0.3 * camera_loss

            loss = total_loss

        return {
            'loss': loss,
            'beat_logits': beat_logits,
            'mood_logits': mood_logits,
            'camera_logits': camera_logits,
        }

## 2. Data Preparation

In [7]:
class ScriptDataset(Dataset):
    """Dataset for script segmentation training."""

    def __init__(
        self,
        texts: List[str],
        labels: List[str],
        moods: List[str],
        cameras: List[str],
        tokenizer,
        max_length: int = 256
    ):
        self.texts = texts
        self.labels = labels
        self.moods = moods
        self.cameras = cameras
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        mood = self.moods[idx]
        camera = self.cameras[idx]

        # Tokenize
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )

        # Create label sequence
        label_id = LABEL_TO_ID.get(label, 0)
        label_ids = [-100] * self.max_length
        label_ids[1] = label_id

        mood_id = MOOD_TO_ID.get(mood, 0)
        camera_id = CAMERA_TO_ID.get(camera, 0)

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label_ids, dtype=torch.long),
            'mood_labels': torch.tensor(mood_id, dtype=torch.long),
            'camera_labels': torch.tensor(camera_id, dtype=torch.long),
        }

In [8]:
# Load preprocessed data
df_sequences = pd.read_parquet(f"{OUTPUT_PATH}/sequence_labels.parquet")

print(f"✅ Loaded {len(df_sequences):,} sequences")
print(f"\nLabel distribution:")
print(df_sequences['label'].value_counts())

✅ Loaded 130,055 sequences

Label distribution:
label
I-BEAT    105775
B-BEAT     24280
Name: count, dtype: int64


In [9]:
# Prepare data
texts = df_sequences['text'].tolist()
labels = df_sequences['label'].tolist()
moods = df_sequences['mood'].tolist()
cameras = df_sequences['camera'].tolist()

# Train/val split
train_texts, val_texts, train_labels, val_labels, train_moods, val_moods, train_cameras, val_cameras = train_test_split(
    texts, labels, moods, cameras,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print(f"Training samples: {len(train_texts):,}")
print(f"Validation samples: {len(val_texts):,}")

Training samples: 104,044
Validation samples: 26,011


In [10]:
# Initialize tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Create datasets
train_dataset = ScriptDataset(
    train_texts, train_labels, train_moods, train_cameras,
    tokenizer, max_length=256
)

val_dataset = ScriptDataset(
    val_texts, val_labels, val_moods, val_cameras,
    tokenizer, max_length=256
)

print(f"✅ Created datasets")
print(f"Sample input shape: {train_dataset[0]['input_ids'].shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

✅ Created datasets
Sample input shape: torch.Size([256])


## 3. Training

In [11]:
# Initialize model
transformers.modeling_utils.PreTrainedModel._can_set_experts_implementation = classmethod(lambda cls: False)
config = DistilBertConfig.from_pretrained('distilbert-base-uncased')

model = ScriptSegmentationModel(
    config,
    num_labels=len(LABEL_TO_ID),
    num_moods=len(MOOD_TO_ID),
    num_cameras=len(CAMERA_TO_ID)
)

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Total parameters: 66,763,284
Trainable parameters: 66,763,284


In [28]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # unpack multi-task outputs
    if isinstance(predictions, tuple):
        beat_logits = predictions[0]   # first task
    else:
        beat_logits = predictions

    if isinstance(labels, tuple):
        beat_labels = labels[0]        # first task
    else:
        beat_labels = labels

    preds = np.argmax(beat_logits, axis=-1)

    # flatten
    preds = preds.reshape(-1)
    beat_labels = beat_labels.reshape(-1)

    # ignore -100
    mask = beat_labels != -100
    preds = preds[mask]
    beat_labels = beat_labels[mask]

    return {
        "f1": f1_score(beat_labels, preds, average="weighted")
    }

In [29]:
# Training arguments
training_args = TrainingArguments(
    output_dir=f"{MODEL_PATH}/checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir=f"{MODEL_PATH}/logs",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to="none",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [30]:
class CustomTrainer(Trainer):
    """Custom trainer to handle multi-task outputs."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        mood_labels = inputs.pop("mood_labels", None)
        camera_labels = inputs.pop("camera_labels", None)

        outputs = model(
            **inputs,
            labels=labels,
            mood_labels=mood_labels,
            camera_labels=camera_labels
        )

        loss = outputs['loss']

        return (loss, outputs) if return_outputs else loss

In [31]:
# Create trainer
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("✅ Trainer initialized")
print(f"\n🚀 Starting training...")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Total steps: {len(train_dataset) // training_args.per_device_train_batch_size * training_args.num_train_epochs}")

✅ Trainer initialized

🚀 Starting training...
   Epochs: 5
   Batch size: 16
   Total steps: 32510


In [32]:
# Train!
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.800600,1.196526,0.836685
2,0.838962,1.083512,0.840156
3,0.819690,1.136471,0.838601
4,0.708955,1.296553,0.838139
5,0.665266,1.364863,0.839230


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=32515, training_loss=0.7608999315086152, metrics={'train_runtime': 485.737, 'train_samples_per_second': 1070.991, 'train_steps_per_second': 66.94, 'total_flos': 3.430289488564224e+16, 'train_loss': 0.7608999315086152, 'epoch': 5.0})

In [33]:
# Evaluate
print("\n📊 Evaluating model...")
eval_results = trainer.evaluate()
print(f"\nEvaluation Results:")
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")


📊 Evaluating model...



Evaluation Results:
  eval_loss: 1.0835
  eval_f1: 0.8402
  eval_runtime: 4.9539
  eval_samples_per_second: 5250.6000
  eval_steps_per_second: 164.1130
  epoch: 5.0000


## 4. Save Model

In [34]:
# Save model and tokenizer
final_model_path = f"{MODEL_PATH}/final_model"

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

# Save label mappings
mappings = {
    'LABEL_TO_ID': LABEL_TO_ID,
    'ID_TO_LABEL': ID_TO_LABEL,
    'MOOD_TO_ID': MOOD_TO_ID,
    'ID_TO_MOOD': ID_TO_MOOD,
    'CAMERA_TO_ID': CAMERA_TO_ID,
    'ID_TO_CAMERA': ID_TO_CAMERA,
}

with open(f"{final_model_path}/label_mappings.json", 'w') as f:
    json.dump(mappings, f, indent=2)

print(f"✅ Model saved to: {final_model_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to: /content/drive/MyDrive/playwright_data/model/final_model


## 5. Test Inference

In [35]:
# Test the model
model.eval()

sample_text = """
INT. ABANDONED WAREHOUSE - NIGHT

Rain hammers against broken windows. DETECTIVE MAYA CHEN steps through the doorway, flashlight cutting through darkness.

MAYA
(whispered)
I know you're here, Marcus.

A SHADOW shifts behind rusted machinery. MARCUS VALE emerges, hands raised.

MARCUS
You shouldn't have come alone.
"""

# Tokenize
inputs = tokenizer(
    sample_text,
    return_tensors='pt',
    truncation=True,
    max_length=256,
    padding='max_length'
).to(device)

# Predict
with torch.no_grad():
    outputs = model(**inputs)

# Get predictions
mood_pred = torch.argmax(outputs['mood_logits'], dim=-1).item()
camera_pred = torch.argmax(outputs['camera_logits'], dim=-1).item()

print(" Sample Inference:")
print(f"\nInput: {sample_text[:100]}...")
print(f"\n Predicted Mood: {ID_TO_MOOD[mood_pred]}")
print(f" Predicted Camera: {ID_TO_CAMERA[camera_pred]}")

🎬 Sample Inference:

Input: 
INT. ABANDONED WAREHOUSE - NIGHT

Rain hammers against broken windows. DETECTIVE MAYA CHEN steps th...

🎭 Predicted Mood: tense
📷 Predicted Camera: medium shot


## Summary

In [36]:
print("="*50)
print("📊 MODEL TRAINING COMPLETE")
print("="*50)
print(f"\n📁 Model saved to: {final_model_path}")
print(f"\n📈 Final Metrics:")
for key, value in eval_results.items():
    print(f"  • {key}: {value:.4f}")
print(f"\n✅ Ready for inference (Notebook 04)")

📊 MODEL TRAINING COMPLETE

📁 Model saved to: /content/drive/MyDrive/playwright_data/model/final_model

📈 Final Metrics:
  • eval_loss: 1.0835
  • eval_f1: 0.8402
  • eval_runtime: 4.9539
  • eval_samples_per_second: 5250.6000
  • eval_steps_per_second: 164.1130
  • epoch: 5.0000

✅ Ready for inference (Notebook 04)


---
## Next Steps

Model trained and saved:
- Model weights: `final_model/`
- Tokenizer: `final_model/`
- Label mappings: `final_model/label_mappings.json`

**Proceed to `04_inference.ipynb`** to:
1. Load the trained model
2. Create inference pipeline
3. Export for production use